# Khảo sát phân phối dữ liệu phòng chat & Xác định Hot Partition

Notebook này nhằm mục đích đọc dữ liệu từ Cassandra bằng PySpark để thống kê số lượng tin nhắn theo từng phòng chat, từ đó xác định xem có phòng chat nào bị lệch dữ liệu quá mức (Hot Partition) gây ảnh hưởng đến hiệu năng hệ thống hay không.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Thiết lập package kết nối Cassandra khi khởi chạy Spark
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.datastax.spark:spark-cassandra-connector_2.12:3.4.1 pyspark-shell'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, min, max

print("Đang khởi động Spark Session...")
spark = SparkSession.builder \
    .appName("EDA_HotPartition_Analysis") \
    .config("spark.cassandra.connection.host", os.getenv("CASSANDRA_HOST", "cassandra-source")) \
    .config("spark.cassandra.connection.port", os.getenv("CASSANDRA_PORT", "9042")) \
    .getOrCreate()

print("Spark Session đã sẵn sàng!")

In [ ]:
# Đọc dữ liệu từ chat_system.chat_table trong Cassandra
df = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="chat_table", keyspace="chat_system") \
    .load()

total_rows = df.count()
print(f"Tổng số tin nhắn hiện tại trong Cassandra: {total_rows}")

In [ ]:
# Kiểm tra các cột dữ liệu và dòng trống (null)
df.printSchema()

for col_name in df.columns:
    null_count = df.filter(col(col_name).isNull()).count()
    print(f"Cột '{col_name}' có {null_count} dòng null.")

In [ ]:
# Xác định khoảng thời gian (range) của dữ liệu tin nhắn
ts_range = df.select(min('timestamp').alias('earliest'), max('timestamp').alias('latest')).collect()[0]
print(f"Tin nhắn sớm nhất: {ts_range['earliest']}")
print(f"Tin nhắn mới nhất: {ts_range['latest']}")

In [ ]:
# Thống kê số lượng tin nhắn theo từng phòng chat
room_counts = df.groupBy("room_id") \
    .agg(count("message_id").alias("message_count")) \
    .orderBy(col("message_count").desc()) \
    .toPandas()

print("Phân bổ tin nhắn trên top 10 phòng chat:")
print(room_counts.head(10))

In [ ]:
# Trực quan hóa dữ liệu bằng biểu đồ cột (Bar Chart)
plt.figure(figsize=(10, 6))
plt.bar(room_counts["room_id"].head(10), room_counts["message_count"].head(10), color='salmon')
plt.title("Phân phối số lượng tin nhắn theo từng phòng (Top 10)")
plt.xlabel("Room ID")
plt.ylabel("Số lượng tin nhắn")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### Kết luận khảo sát:
- Phòng chat **room_999** chứa số lượng tin nhắn cực lớn so với các phòng khác (chiếm tới ~80% tổng số tin nhắn của toàn bộ hệ thống).
- Với mô hình dữ liệu cũ của Cassandra (sử dụng duy nhất `room_id` làm Partition Key), toàn bộ tin nhắn của `room_999` sẽ bị dồn vào một partition vật lý duy nhất trên một node duy nhất.
- Đây chính là tác nhân gây ra lỗi **Hot Partition** (phân vùng nóng) làm nghẽn hệ thống khi phòng chat này có hoạt động gửi nhận tin nhắn liên tục.